# ============================================================
# TURKQUISH — GENERATE URL-TRANSFORMER BACKEND ARTIFACTS
# ============================================================
# Input:
#   url_transformer_best.pt
#
# Output:
#   app/artifacts/url_transformer/
#     url_transformer.pt
#     char_vocab.json
#     transformer_config.json
#     label_encoder.json
#     checkpoint_inspection.json
#
# This matches your notebook URL-Transformer settings:
#   MAX_LEN=200, EMBED_DIM=64, heads=4, layers=3, d_ff=128, dropout=0.1
# ============================================================

In [ ]:
import os
import re
import json
import math
import shutil
from pathlib import Path
from datetime import datetime

import torch
import torch.nn as nn


# ============================================================
# 1. PATHS
# ============================================================

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"

MODEL_IN = os.path.join(DATA_DIR, "url_transformer_best.pt")

OUT_DIR = Path(DATA_DIR) / "artifacts_5class" / "url_transformer"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("URL-TRANSFORMER ARTIFACT EXPORT")
print("=" * 80)
print("Input model:", MODEL_IN)
print("Output dir :", OUT_DIR)

if not os.path.exists(MODEL_IN):
    raise FileNotFoundError(f"Model not found: {MODEL_IN}")


# ============================================================
# 2. EXACT VOCABULARY FROM YOUR NOTEBOOK
# ============================================================

PAD_IDX = 0
UNK_IDX = 1

CHARS = (
    "abcdefghijklmnopqrstuvwxyz0123456789"
    ".-_/:?=&%@+~#!$*()[]{}|\\;,<>\"' çğıöşüâî"
)

# Training used:
# char2idx = {c: i + 2 for i, c in enumerate(CHARS)}
# VOCAB = len(char2idx) + 2
#
# For backend, include <PAD> and <UNK> inside char2idx so len(char2idx) == VOCAB.
char2idx = {
    "<PAD>": PAD_IDX,
    "<UNK>": UNK_IDX,
}
for i, ch in enumerate(CHARS):
    char2idx[ch] = i + 2

idx2char = {str(v): k for k, v in char2idx.items()}

VOCAB_SIZE = len(char2idx)

print("\n[1] Vocabulary")
print("CHARS length:", len(CHARS))
print("Vocab size  :", VOCAB_SIZE)
print("PAD_IDX     :", PAD_IDX)
print("UNK_IDX     :", UNK_IDX)


# ============================================================
# 3. DEFAULT CONFIG FROM YOUR NOTEBOOK
# ============================================================

MAX_LEN = 200
D_MODEL = 64
N_HEADS = 4
N_LAYERS = 3
D_FF = 128
DROPOUT = 0.1
N_CLASSES = 5

# Recommended for backend/reporting.
# This is the canonical TUMC manuscript mapping:
# 0=benign, 1=phishing, 2=malware, 3=scam, 4=other_malicious
#
# If your URL-Transformer was trained using string labels sorted alphabetically,
# switch LABEL_ORDER_MODE to "alpha".
LABEL_ORDER_MODE = "canonical_class_enc"   # options: "canonical_class_enc", "alpha"

if LABEL_ORDER_MODE == "canonical_class_enc":
    CLASS_NAMES = ["benign", "phishing", "malware", "scam", "other_malicious"]
else:
    CLASS_NAMES = ["benign", "malware", "other_malicious", "phishing", "scam"]

print("\n[2] Config")
print("MAX_LEN :", MAX_LEN)
print("D_MODEL :", D_MODEL)
print("N_HEADS :", N_HEADS)
print("LAYERS  :", N_LAYERS)
print("D_FF    :", D_FF)
print("DROPOUT :", DROPOUT)
print("Classes :", CLASS_NAMES)


# ============================================================
# 4. LOAD CHECKPOINT AND NORMALIZE STATE_DICT
# ============================================================

print("\n[3] Loading checkpoint...")

checkpoint = torch.load(MODEL_IN, map_location="cpu")

def extract_state_dict(obj):
    """
    Supports:
      - raw state_dict
      - {'state_dict': ...}
      - {'model_state_dict': ...}
      - {'model': ...}
    """
    if isinstance(obj, dict):
        for key in ["state_dict", "model_state_dict", "model", "net"]:
            if key in obj and isinstance(obj[key], dict):
                return obj[key], key
        # If it already looks like a raw state_dict
        if any(isinstance(v, torch.Tensor) for v in obj.values()):
            return obj, "raw_state_dict"
    raise ValueError(
        "Could not extract model state_dict from checkpoint. "
        "The .pt may contain a full model object instead of a state_dict."
    )

state_dict, state_source = extract_state_dict(checkpoint)

# Remove DataParallel prefix if present
clean_state = {}
for k, v in state_dict.items():
    nk = k.replace("module.", "", 1) if k.startswith("module.") else k
    clean_state[nk] = v

state_dict = clean_state

print("Checkpoint source:", state_source)
print("Number of tensors:", len(state_dict))
print("First 15 keys:")
for k in list(state_dict.keys())[:15]:
    print("  ", k, tuple(state_dict[k].shape) if hasattr(state_dict[k], "shape") else type(state_dict[k]))


# ============================================================
# 5. INSPECT SHAPES TO VERIFY CONFIG
# ============================================================

def find_key(patterns):
    for p in patterns:
        for k in state_dict.keys():
            if re.search(p, k):
                return k
    return None

embed_key = find_key([
    r"^embed\.weight$",
    r"embedding\.weight$",
    r"emb\.weight$"
])

head_last_key = None
for k in state_dict.keys():
    if re.search(r"head\.\d+\.weight$", k) or re.search(r"classifier\.\d+\.weight$", k) or re.search(r"fc\.weight$", k):
        tensor = state_dict[k]
        if hasattr(tensor, "shape") and len(tensor.shape) == 2:
            # final classifier normally has output size <= 10
            if tensor.shape[0] <= 20:
                head_last_key = k

layer_ids = []
for k in state_dict.keys():
    m = re.search(r"encoder\.layers\.(\d+)\.", k)
    if m:
        layer_ids.append(int(m.group(1)))

inspection = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "checkpoint_path": MODEL_IN,
    "state_source": state_source,
    "n_tensors": len(state_dict),
    "embed_key": embed_key,
    "head_last_key": head_last_key,
    "detected_layers": sorted(set(layer_ids)),
}

if embed_key:
    saved_vocab, saved_d = state_dict[embed_key].shape
    inspection["saved_vocab_size"] = int(saved_vocab)
    inspection["saved_d_model"] = int(saved_d)

    print("\n[4] Detected from checkpoint")
    print("Embedding key:", embed_key)
    print("Saved vocab  :", saved_vocab)
    print("Saved d_model:", saved_d)

    if saved_vocab != VOCAB_SIZE:
        raise RuntimeError(
            f"Vocab mismatch: checkpoint expects {saved_vocab}, "
            f"but generated vocab has {VOCAB_SIZE}.\n"
            "Do not continue with a different vocabulary. "
            "You must recreate the exact CHARS string used during training."
        )

    if saved_d != D_MODEL:
        raise RuntimeError(
            f"D_MODEL mismatch: checkpoint expects {saved_d}, config has {D_MODEL}."
        )
else:
    print("\nWARNING: Could not detect embedding weight key.")

if head_last_key:
    saved_n_classes = int(state_dict[head_last_key].shape[0])
    inspection["saved_n_classes"] = saved_n_classes

    print("Classifier key:", head_last_key)
    print("Saved classes :", saved_n_classes)

    if saved_n_classes != N_CLASSES:
        raise RuntimeError(
            f"Class count mismatch: checkpoint expects {saved_n_classes}, "
            f"config has {N_CLASSES}."
        )
else:
    print("WARNING: Could not detect final classifier key.")

if layer_ids:
    detected_layers = max(layer_ids) + 1
    inspection["saved_n_layers"] = detected_layers
    print("Detected layers:", detected_layers)

    if detected_layers != N_LAYERS:
        raise RuntimeError(
            f"Layer mismatch: checkpoint has {detected_layers}, config has {N_LAYERS}."
        )


# ============================================================
# 6. DEFINE MODEL CLASS FOR VALIDATION
# ============================================================

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=MAX_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class URLTransformer(nn.Module):
    def __init__(
        self,
        vocab,
        n_classes,
        d=D_MODEL,
        heads=N_HEADS,
        layers=N_LAYERS,
        d_ff=D_FF,
        drop=DROPOUT,
        pad_idx=PAD_IDX,
        max_len=MAX_LEN,
    ):
        super().__init__()
        self.pad_idx = pad_idx
        self.d_model = d

        self.embed = nn.Embedding(vocab, d, padding_idx=pad_idx)
        self.pos = PositionalEncoding(d, max_len=max_len)

        enc = nn.TransformerEncoderLayer(
            d,
            heads,
            d_ff,
            drop,
            activation="gelu",
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc, layers)
        self.dropout = nn.Dropout(drop)

        self.head = nn.Sequential(
            nn.Linear(d * 2, d),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(d, n_classes),
        )

    def forward(self, x):
        mask = x == self.pad_idx
        h = self.pos(self.embed(x) * math.sqrt(self.d_model))
        h = self.encoder(h, src_key_padding_mask=mask)

        hm = h.masked_fill(mask.unsqueeze(-1), 0).sum(1) / (
            (~mask).sum(1, keepdim=True).clamp(min=1)
        )
        hx = h.masked_fill(mask.unsqueeze(-1), -1e9).max(1).values

        return self.head(self.dropout(torch.cat([hm, hx], 1)))


def encode_url(url, max_len=MAX_LEN):
    url = str(url).lower()[:max_len]
    ids = [char2idx.get(ch, UNK_IDX) for ch in url]
    ids += [PAD_IDX] * (max_len - len(ids))
    return torch.tensor([ids], dtype=torch.long)


# ============================================================
# 7. VALIDATE MODEL LOAD
# ============================================================

print("\n[5] Validating model load...")

model = URLTransformer(
    vocab=VOCAB_SIZE,
    n_classes=N_CLASSES,
    d=D_MODEL,
    heads=N_HEADS,
    layers=N_LAYERS,
    d_ff=D_FF,
    drop=DROPOUT,
    pad_idx=PAD_IDX,
    max_len=MAX_LEN,
)

missing, unexpected = model.load_state_dict(state_dict, strict=False)

print("Missing keys   :", missing)
print("Unexpected keys:", unexpected)

# Strict validation: allow no missing/unexpected for backend.
if missing or unexpected:
    raise RuntimeError(
        "Model state_dict does not exactly match this URLTransformer architecture.\n"
        "Most likely causes:\n"
        "  1. The .pt file is from CNNBiLSTM, not URLTransformer.\n"
        "  2. The URLTransformer architecture used different head/dropout/layers.\n"
        "  3. The checkpoint key names differ from the backend model.\n"
        "Fix the architecture/config before exporting."
    )

model.eval()

with torch.no_grad():
    sample_url = "https://garanti-giris.xyz/hesap-dogrula"
    logits = model(encode_url(sample_url))
    probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

print("\nSample URL:", sample_url)
print("Probabilities:")
for label, p in zip(CLASS_NAMES, probs):
    print(f"  {label:<18s}: {float(p):.4f}")


# ============================================================
# 8. SAVE BACKEND ARTIFACTS
# ============================================================

print("\n[6] Saving artifacts...")

# Save normalized raw state_dict for backend.
# Backend supports raw state_dict cleanly.
torch.save(state_dict, OUT_DIR / "url_transformer.pt")

char_vocab = {
    "version": "turkquish-char-vocab-v1",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "pad_token": "<PAD>",
    "unk_token": "<UNK>",
    "pad_idx": PAD_IDX,
    "unk_idx": UNK_IDX,
    "chars": CHARS,
    "char2idx": char2idx,
    "idx2char": idx2char,
    "vocab_size": VOCAB_SIZE,
    "important_note": (
        "This vocabulary must exactly match the character-to-index mapping "
        "used when url_transformer_best.pt was trained."
    ),
}

with open(OUT_DIR / "char_vocab.json", "w", encoding="utf-8") as f:
    json.dump(char_vocab, f, indent=2, ensure_ascii=False)

transformer_config = {
    "model_name": "URLTransformer",
    "model_version": "url-transformer-5class-v1",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "task": "five_class_url_character_classification",
    "max_len": MAX_LEN,
    "n_classes": N_CLASSES,
    "vocab_size": VOCAB_SIZE,
    "embed_dim": D_MODEL,
    "d_model": D_MODEL,
    "heads": N_HEADS,
    "layers": N_LAYERS,
    "d_ff": D_FF,
    "dropout": DROPOUT,
    "pad_idx": PAD_IDX,
    "unk_idx": UNK_IDX,
    "activation": "gelu",
    "pooling": "mean_pool_plus_max_pool",
    "input": "raw_url_lowercase_truncated_to_max_len",
    "url_only": True,
    "forbidden_runtime_actions": [
        "DNS lookup",
        "WHOIS lookup",
        "HTML retrieval",
        "webpage screenshot",
        "third-party reputation query",
        "live webpage fetch",
    ],
}

with open(OUT_DIR / "transformer_config.json", "w", encoding="utf-8") as f:
    json.dump(transformer_config, f, indent=2, ensure_ascii=False)

label_encoder = {
    "label_type": LABEL_ORDER_MODE,
    "created_at": datetime.utcnow().isoformat() + "Z",
    "classes": CLASS_NAMES,
    "model_classes_label_order": CLASS_NAMES,
    "id_to_label": {str(i): label for i, label in enumerate(CLASS_NAMES)},
    "label_to_id": {label: i for i, label in enumerate(CLASS_NAMES)},
    "probability_order_note": (
        "URL-Transformer softmax probabilities follow this exact class order. "
        "If the model was trained with a different label order, change CLASS_NAMES "
        "and regenerate this file."
    ),
}

with open(OUT_DIR / "label_encoder.json", "w", encoding="utf-8") as f:
    json.dump(label_encoder, f, indent=2, ensure_ascii=False)

inspection["exported_files"] = [
    "url_transformer.pt",
    "char_vocab.json",
    "transformer_config.json",
    "label_encoder.json",
]
inspection["label_order"] = CLASS_NAMES
inspection["config"] = transformer_config

with open(OUT_DIR / "checkpoint_inspection.json", "w", encoding="utf-8") as f:
    json.dump(inspection, f, indent=2, ensure_ascii=False)

print("Saved:", OUT_DIR / "url_transformer.pt")
print("Saved:", OUT_DIR / "char_vocab.json")
print("Saved:", OUT_DIR / "transformer_config.json")
print("Saved:", OUT_DIR / "label_encoder.json")
print("Saved:", OUT_DIR / "checkpoint_inspection.json")


# ============================================================
# 9. ZIP OPTIONAL
# ============================================================

zip_path = shutil.make_archive(str(OUT_DIR), "zip", root_dir=str(OUT_DIR))

print("\n" + "=" * 80)
print("URL-TRANSFORMER EXPORT COMPLETE ✅")
print("=" * 80)
print("Output folder:", OUT_DIR)
print("ZIP file     :", zip_path)
print("\nCopy this folder into your backend as:")
print("backend/app/artifacts/url_transformer/")

URL-TRANSFORMER ARTIFACT EXPORT
Input model: /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/url_transformer_best.pt
Output dir : /content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May/artifacts_5class/url_transformer

[1] Vocabulary
CHARS length: 75
Vocab size  : 77
PAD_IDX     : 0
UNK_IDX     : 1

[2] Config
MAX_LEN : 200
D_MODEL : 64
N_HEADS : 4
LAYERS  : 3
D_FF    : 128
DROPOUT : 0.1
Classes : ['benign', 'phishing', 'malware', 'scam', 'other_malicious']

[3] Loading checkpoint...
Checkpoint source: raw_state_dict
Number of tensors: 42
First 15 keys:
   embed.weight (77, 64)
   pos.pe (1, 200, 64)
   encoder.layers.0.self_attn.in_proj_weight (192, 64)
   encoder.layers.0.self_attn.in_proj_bias (192,)
   encoder.layers.0.self_attn.out_proj.weight (64, 64)
   encoder.layers.0.self_attn.out_proj.bias (64,)
   encoder.layers.0.linear1.weight (128, 64)
   encoder.layers.0.linear1.bias (128,)
   encoder.layers.0.linear2.weight (64, 128)
   encoder.layers.0.linear2